In [7]:
import pandas as pd
from IPython.core.magic import register_cell_magic
from google_play_scraper import reviews_all
from utils import get_ngram_counts, review_prep
import time
from pathlib import Path

In [2]:
@register_cell_magic
def skip(line, cell):
    return

data scraping

In [3]:
%%skip
languages = ["pl"]
apps_ids = ["pl.mapa_turystyczna.app", "de.komoot.android", "com.strava", "com.alltrails.alltrails",
            "com.meteoblue.droid", "menion.android.locus", "cz.seznam.mapy", "com.pagasolutions.emergencycall_pl",
            "info.burzowo", "com.windyty.android", "app.com.example.szymi.myapplication", "pl.traseo2", "no.nrk.yr",
            "pl.janpogocki.icm.pogoda", "com.verdant.kgp", "com.verdant.wkt", "org.peakfinder.area.alps",
            "com.google.android.apps.maps"]

parsed_list = []

for count, app_id in enumerate(apps_ids, start=1):
    print(f"{app_id} is running. {count}/{len(apps_ids)}")
    time.sleep(7)
    for lang in languages:
        parsed_dict = reviews_all(app_id=app_id, sleep_milliseconds=1200, lang=lang)
        partial_df = pd.DataFrame(parsed_dict).assign(appId=app_id, language=lang)
        partial_df = partial_df.drop(columns=["userImage", "userName"])
        parsed_list.append(partial_df)


df = pd.concat(parsed_list, ignore_index=True)

df.to_csv(Path("data") / "reviews_raw.csv", index=False)
print("success")

loading data from drive if parsing finished earlier

In [11]:
df = pd.read_csv(
    Path("data") / "reviews_raw.csv",
    dtype={
        "appId": str,
        "language": str,
        "reviewId": str,
        "score": "int8",
        "content": str,
        "thumbsUpCount": "int32",
        "reviewCreatedVersion": str,
        "replyContent": str,
        "appVersion": str,
    },
    parse_dates=["at", "repliedAt"],
)
apps_ids = df["appId"].unique()
languages = df["language"].unique()

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 157580 entries, 0 to 157579
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   reviewId              157580 non-null  str           
 1   content               157411 non-null  str           
 2   score                 157580 non-null  int8          
 3   thumbsUpCount         157580 non-null  int32         
 4   reviewCreatedVersion  153056 non-null  str           
 5   at                    157580 non-null  datetime64[us]
 6   replyContent          16143 non-null   str           
 7   repliedAt             16143 non-null   datetime64[us]
 8   appVersion            153056 non-null  str           
 9   appId                 157580 non-null  str           
 10  language              157580 non-null  str           
dtypes: datetime64[us](2), int32(1), int8(1), str(7)
memory usage: 11.6 MB


lemmatization

In [ ]:
reviews_lemmatized = review_prep(df, score_range=(0, 5))

reviews_lemmatized.to_csv(Path("data") / "reviews_with_lemmas.csv", index=False)

counting ngrams

In [ ]:
counted_ngrams = []
for app_id in apps_ids:
    app_ngram = (
        get_ngram_counts(reviews_lemmatized[reviews_lemmatized["appId"] == app_id])
        .assign(appId=app_id)
        .reset_index(names="keywords")
    )
    counted_ngrams.append(app_ngram)
all_ngrams = pd.concat(counted_ngrams, ignore_index=True)

all_ngrams.to_csv(Path("data") / "reviews_ngram_counts.csv", index=False)